In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install timm -q

timm is a library of pretrained vision models and midas internally uses it

# Imports

In [3]:
# imports
import torch
import cv2
import numpy as np
import os

# Settings 

In [4]:
# settings change
Video_path= ""
Output_folder="/kaggle/working/output"
Strength= 18
MAX_SECONDS= 60 

os.makedirs(Output_folder, exist_ok= True)
device= "cuda" if torch.cuda.is_available() else "cpu"
print("using device: ", device)

using device:  cuda


* paste the copy path of input video in Video_path.
* our all outputs will be saved into Output_folder i.e. Kaggle output directory.
* Strength is how far pixel should shift, higher value= stronger 3d effect but also some distortion.
* MAX_SECONDS: how long your input should be, it trim the input video upto that specific seconds only.


In [ ]:
#output selection
print("Select Output Type")
print("1-> for Anaglyph Output (Red-Cyan Glasses)")
print("2-> for SBS Output (VR Headset)")
print("3-> for Depth Map Output Only")
print("4-> for All 3 Outputs")

Output= input("\n Enter Choice (1/2/3/4) : ").strip()
if Output not in ["1","2","3","4"]:
    print("Invalid Choice, Defaulting to 4 (All Outputs)")
    Output= 4

type_names= {"1": "Anaglyph", "2": "SBS", "3": "Depth Map only", "4": "All 3 Outputs"}

print(f"\n SELECTED : {type_names[Output]}")

Select Output Type
1-> for Anaglyph Output (Red-Cyan Glasses)
2-> for SBS Output (VR Headset)
3-> for Depth Map Output Only
4-> for All 3 Outputs


# Load MiDaS

In [ ]:
#load midas
model= torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid", trust_repo= True)
model.to(device)
model.eval()

transforms= torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo= True)
transform= transforms.dpt_transform
print("Model loaded on: ", device)

* MiDaS is a pytorch model developed by intel, we are using DPT_Hybrid version here
* model.eval() turns model into inference mode from training mode

# The Depth Function

In [ ]:
#the depth function
prev_depth= None
smooth= 0.75

def get_depth(frame):
    global prev_depth

    small= cv2.resize(frame, (640, 480))
    rgb= cv2.cvtCOLOR(small, cv2.COLOR_BGR2RGB)
    input_tensor= transform(rgb).to(device)
    #inferencing input frame hrough model
    with torch.no_grad():  #only inference
        depth= model(input_tensor) 
        depth= torch.nn.functional.interpolate( #function to resize tensors
            depth.unsqueeze(1),  #adds dimension
            size= frame.shape[:2],
            mode= 'bicubic',
            align_corners= False
        ).squeeze().cpu().numpy()
#normalization of frames in 0-1
depth= (depth-depth.min())/ (depth.max()- depth.min()+ 1e-8)
if prev_depth is not None():
    depth= (smooth * depth) + ((1- smooth) * prev_depth)
prev_depth= depth
return depth
    



* here we are inferencing our frme form input video through the model, but before we are resizing the frame suitable for our midas model like resizing, converting BGR into RGB, adding dimensions, and most importantly converting our input frame into a tensor
* the normalization function scales all values of the tensor array to 0-1, The 1e-8 (a tiny number) is added to the denominator to prevent division by zero on completely flat scenes.
* The smoothing block is temporal smoothing. On the first frame prev_depth is None so we skip it. From second frame onward, we blend current depth with previous depth using the SMOOTH weight. 0.75 means 75% current frame, 25% previous frame. it prevents frames from flickering.

# Pixel Shifting

In [ ]:
# pixel shifting
def shift_pixels(frame, depth, direction):
    h,w= frame.shape[:2]
    shift= (depth* Strength* direction).astype(np.int32)
    ys, xs= np.mgrid[0:h, 0:w]
    new_xs= np.clip(xs + shift, 0, w-1)
    out= np.zeros_like(frame)
    out[ys, new_xs]= frame[ys, xs]
    return out

def make_anaglyph(frame, depth):
    left= shift_pixels(frame, depth, +1)
    right= shift_pixels(frame, depth, -1)
    anaglyph= np.zeros_like(frame)
    anaglyph[:,:,2]= left[:,:,2]
    anaglyph[:,:,1]= right[:,:, 1]
    anaglyph[:,:,0]= right[:,:, 0]
    return anaglyph

def make_sbs(frame, depth):
    left= shift_pixels(frame, depth, +1)
    right= shift_pixels(frame, depth, -1)
    return np.concatenate([left, right], axis)

* for every pixel multiply its depth value (0-1) by strength (mentioned in settings) to get how many pixels to shift
* np.mgrid[0:h, 0:w]: creates two 2d grids, one contains y coordinate of every pixel, one contains x coordinate of every pixel
* new_xs = np.clip(xs + shifts, 0, w-1) adds the shift to every x coordinate simultaneously. clip ensures no coordinate goes out of bounds — beyond left or right edge of the image.
* For make_sbs — np.concatenate([left, right], axis=1) joins two arrays side by side. axis=1 means concatenate along the width dimension (horizontal). The result has double the width.

# Main Loop

In [ ]:
cap= cv2.VideoCapture(Video_path)
fps= cap.get(cv2.CAP_PROP_FPS)
w= int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h= int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
max_frames= int(fps* MAX_SECONDS)

print(f"Video: {w}*{h} at {fps: .0f} FPS| Processing {max_frames} frames")
print(f"\n Processing {max_frames} frames  - Output {type_names[Output]}")

fourcc= cv2.VideoWriter_fourcc(*'mp4v')

out_ana= cv2.VideoWriter(f"{Output_folder}/anaglyph.mp4", fourcc, fps, (w,h))
out_sbs= cv2.VideoWriter(f"{Output_folder}/sidebyside.mp4", fourcc, fps, (w*2,h))
out_dep= cv2.VideoWriter(f"{Output_folder}/depthmap.mp4", fourcc, fps, (w,h))

frame_num= 0
while frame_num< max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    depth= get_depth(frame)
    anaglyph= make_anaglyph(frame, depth)
    sbs= make_sbs(frame, depth)
    depth_colored= cv2.applyColorMap((depth*255).astype(np.uint8), cv2.COLORMAP_MAGMA)
   
    out_ana.write(anaglyph)
    out_sbs.write(sbs)
    out_dep.write(depth_colored)


    frame_num+=1
    if frame_num%20== 0:
        print(f"Processed {frame_num}/ {max_frames} frames")

cap.release()

if out_ana is not None: out_ana.release()
if out_sbs is not None: out_sbs.release()
if out_dep is not None: out_dep.release()


if OUTPUT_TYPE in ["1", "4"]: print(f"  Anaglyph  → {OUTPUT_FOLDER}/anaglyph.mp4")
if OUTPUT_TYPE in ["2", "4"]: print(f"  SBS       → {OUTPUT_FOLDER}/sidebyside.mp4")
if OUTPUT_TYPE in ["3", "4"]: print(f"  Depth map → {OUTPUT_FOLDER}/depth.mp4")
        

* cv2.VideoWriter_fourcc(*'mp4v') defines the video codec — the compression algorithm for the output file. mp4v is standard MPEG-4 encoding compatible with most players. The * unpacks the string into four separate characters as the function expects
* cv2.VideoWriter(path, fourcc, fps, (width, height)) creates an output video file. Notice SBS gets (w*2, h) because it's double width. If this size doesn't match the frames you write to it, OpenCV silently drops those frames — no error, just missing frames in output. This is a common silent bug.

# add audio (optional)

In [ ]:
def add_audio(original_video, silent_video, output_path):
    cmd = (
        f'ffmpeg -i "{silent_video}" -i "{original_video}" '
        f'-c:v copy -c:a aac -map 0:v:0 -map 1:a:0 '
        f'-shortest "{output_path}" -y -loglevel error'
    )
    os.system(cmd)
    print(f"Audio added → {output_path}")

add_audio(VIDEO_PATH, f"{OUTPUT_FOLDER}/anaglyph.mp4",   f"{OUTPUT_FOLDER}/anaglyph_final.mp4")
add_audio(VIDEO_PATH, f"{OUTPUT_FOLDER}/sidebyside.mp4", f"{OUTPUT_FOLDER}/sidebyside_final.mp4")
add_audio(VIDEO_PATH, f"{OUTPUT_FOLDER}/depth.mp4",      f"{OUTPUT_FOLDER}/depth_final.mp4")

Creative Prompt-to-Image Platform
- Built production-ready image generation tool with advanced controls
- Integrated multiple ControlNet models (edge, depth, pose) for precise generation
- Implemented batch processing, style presets, and generation history
- Deployed live demo with 100+ generated samples showcasing capabilities
Tech: Stable Diffusion 2.1, ControlNet, ComfyUI, Streamlit